# 📖 Notebook 4: Disaster Recovery Drill

## Why This Matters

Netflix runs **Chaos Monkey** — a tool that randomly kills production servers
to ensure their systems can handle failure. Banks run quarterly DR drills.
Microsoft Azure tests failover across entire regions.

The only way to know if your BCDR plan works is to **test it**.

In this notebook, you will simulate a disaster and execute a complete
recovery procedure while measuring your actual RPO and RTO.

## Learning Objectives

- Plan and execute a disaster recovery drill
- Simulate primary database failure
- Execute the full failover procedure
- Measure actual RPO (data lost) and RTO (downtime)
- Generate a DR drill report

In [ ]:
# ── Preflight: FAIL BACK from notebook 2's failover ──────────────────────────
#
# Notebook 2 stops mid-disaster on purpose: the primary is FENCED (its
# container is stopped -- the STONITH step of a real failover) and the standby
# is PROMOTED. That is the state an on-call engineer is actually handed, and
# it is not a state this notebook can run in:
#
#   * port 5432 is dead, so every connection below would fail, and
#   * the promoted node on 55433 is now an INDEPENDENT primary holding writes
#     the fenced node never saw. Simply restarting the fenced node gives you
#     two primaries with diverged data -- textbook split-brain -- and throws
#     the promoted node's writes away without saying so.
#
# So this cell performs a real FAILBACK, in runbook order:
#   1. capture the rows written on the promoted node during the failover
#      window (that divergence is the whole problem),
#   2. un-fence the old primary,
#   3. re-apply the captured rows, so nothing written during the outage is
#      lost -- a failback that drops them is data loss with a success message,
#   4. re-clone the standby from the primary so streaming replication resumes,
#   5. verify: primary writable, standby in recovery AND streaming, and the
#      carried-back rows visible on both.
#
# Production does step 3 with `pg_rewind` plus WAL replay, which preserves
# every table and every row id. We do it at row level because this lab's
# failover-window writes all land in one table (`audit_log`) -- the lesson is
# the same. Calling this when nothing is wrong is a no-op, so it is safe to
# re-run and safe if you never ran notebook 2 at all.
import socket
import subprocess
import time
from pathlib import Path

import psycopg2

PRIMARY_PORT, STANDBY_PORT = 5432, 55433
PRIMARY_SERVICE, STANDBY_SERVICE = "pg-primary", "pg-standby"
PRIMARY_CONTAINER, STANDBY_CONTAINER = "bcdr-pg-primary", "bcdr-pg-standby"
PGDATA_PATH = "/var/lib/postgresql/data"
AUDIT_COLS = "table_name, record_id, action, changed_by, changed_at"


def _lab_root():
    """The directory holding docker-compose.yml, found from wherever we are."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "docker-compose.yml").exists():
            return cand
    raise RuntimeError(f"no docker-compose.yml at or above {here}")


LAB_ROOT = _lab_root()


def _run(cmd, cwd=None, timeout=300):
    return subprocess.run(cmd, cwd=cwd and str(cwd),
                          capture_output=True, text=True, timeout=timeout)


def port_open(port, host="127.0.0.1", timeout=1.0):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def pg_query(port, sql, params=None):
    """Rows from the node on `port`, or None if it is not answering.

    READ-ONLY on purpose: the connection is closed without committing, so
    psycopg2 rolls back. Use it to inspect a node, never to change one --
    `carry_back()` below opens its own autocommit connection for writes."""
    try:
        conn = psycopg2.connect(host="127.0.0.1", port=port, dbname="bcdr_demo",
                                user="demo", password="demo", connect_timeout=3)
    except psycopg2.Error:
        return None
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            return cur.fetchall()
    except psycopg2.Error:
        return None
    finally:
        conn.close()


def node_role(port):
    """'down', 'primary' (writable) or 'standby' (in recovery)."""
    rows = pg_query(port, "SELECT pg_is_in_recovery()")
    if rows is None:
        return "down"
    return "standby" if rows[0][0] else "primary"


def streaming_ok():
    rows = pg_query(PRIMARY_PORT,
                    "SELECT 1 FROM pg_stat_replication WHERE state = 'streaming'")
    return bool(rows)


def topology_ok():
    """The documented shape: writable primary on 5432, streaming standby on 55433."""
    return (node_role(PRIMARY_PORT) == "primary"
            and node_role(STANDBY_PORT) == "standby"
            and streaming_ok())


def wait_topology(timeout=120):
    """Give a cold `docker compose up` time to finish the standby's basebackup
    before we conclude anything is wrong."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        if topology_ok():
            return True
        time.sleep(2)
    return topology_ok()


def audit_rows(port):
    rows = pg_query(port, f"SELECT {AUDIT_COLS} FROM audit_log")
    return None if rows is None else {tuple(r) for r in rows}


def start_primary(timeout=180):
    """Un-fence the old primary and wait until it is writable again."""
    _run(["docker", "compose", "up", "-d", PRIMARY_SERVICE], cwd=LAB_ROOT)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if node_role(PRIMARY_PORT) == "primary":
            return
        time.sleep(2)
    raise RuntimeError(
        "the fenced primary did not come back on port 5432. Reset the lab:\n"
        "    docker compose down -v && docker compose up -d --wait")


def carry_back(rows):
    """Re-apply failover-window rows onto the primary, keeping their original
    timestamps. Row ids are NOT preserved -- they came from the promoted
    node's own sequence. `pg_rewind` would keep them; row-level carry-back
    cannot, which is one honest reason production prefers pg_rewind."""
    if not rows:
        return 0
    conn = psycopg2.connect(host="127.0.0.1", port=PRIMARY_PORT,
                            dbname="bcdr_demo", user="demo", password="demo")
    conn.autocommit = True
    try:
        with conn.cursor() as cur:
            for r in sorted(rows, key=lambda row: row[4]):
                cur.execute(
                    "INSERT INTO audit_log (table_name, record_id, action, "
                    "changed_by, changed_at) VALUES (%s, %s, %s, %s, %s)", r)
    finally:
        conn.close()
    return len(rows)


def standby_volume():
    """The named volume backing the standby's PGDATA (asked, not guessed)."""
    fmt = ('{{range .Mounts}}{{if eq .Destination "' + PGDATA_PATH
           + '"}}{{.Name}}{{end}}{{end}}')
    name = _run(["docker", "inspect", "-f", fmt, STANDBY_CONTAINER]).stdout.strip()
    return name or f"{LAB_ROOT.name}_pg_standby_data"


def rebuild_standby(timeout=300):
    """Re-clone the standby from the primary. A promoted node has switched to
    a new WAL timeline and cannot just reattach; real clusters run pg_rewind
    here, but a lab-sized database is faster to clone from scratch."""
    vol = standby_volume()
    _run(["docker", "compose", "rm", "-sf", STANDBY_SERVICE], cwd=LAB_ROOT)
    _run(["docker", "volume", "rm", "-f", vol])
    # pg_basebackup reuses 'standby_slot' and refuses while the slot still
    # shows a live walsender from the node we just removed.
    deadline = time.time() + 60
    while time.time() < deadline:
        rows = pg_query(PRIMARY_PORT,
                        "SELECT active FROM pg_replication_slots "
                        "WHERE slot_name = 'standby_slot'")
        if not rows or not rows[0][0]:
            break
        time.sleep(1)
    _run(["docker", "compose", "up", "-d", STANDBY_SERVICE], cwd=LAB_ROOT)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if node_role(STANDBY_PORT) == "standby" and streaming_ok():
            return
        time.sleep(2)
    raise RuntimeError(
        "the standby did not come back as a streaming standby. Reset the lab:\n"
        "    docker compose down -v && docker compose up -d --wait")


def failback():
    """Restore the documented topology WITHOUT losing failover-window writes."""
    broken_now = (node_role(PRIMARY_PORT) != "primary"
                  or node_role(STANDBY_PORT) == "primary")
    if not broken_now and wait_topology():
        print("✅ Topology is already the documented one: "
              "primary on 5432, streaming standby on 55433. Nothing to do.")
        return {"action": "none", "carried_back": 0, "carried_markers": []}

    print("⚠️  Topology is not the documented one — running FAILBACK.")
    print(f"    port 5432: {node_role(PRIMARY_PORT):<8} "
          f"port 55433: {node_role(STANDBY_PORT)}")

    # 1. Capture the divergence while the promoted node is still its only home.
    promoted = (audit_rows(STANDBY_PORT)
                if node_role(STANDBY_PORT) == "primary" else None)

    # 2. Un-fence the old primary.
    start_primary()
    print("    ✅ old primary is back on 5432")

    # 3. Carry the failover-window writes forward.
    carried, markers = 0, []
    if promoted is None:
        print("    ·  the promoted node was not reachable — nothing to carry back")
    else:
        divergent = promoted - (audit_rows(PRIMARY_PORT) or set())
        markers = sorted({r[3] for r in divergent if r[3]})
        carried = carry_back(divergent)
        print(f"    ✅ carried {carried} row(s) written during the failover "
              f"window back onto the primary")
        if markers:
            print(f"       markers: {', '.join(markers)}")

    # 4. Re-form the pair.
    #
    #    A standby that was merely stopped or briefly detached -- which is
    #    where a failover that dies half-way leaves you, with no writable node
    #    anywhere -- can reattach on its own now that the primary is back.
    #    Wiping a perfectly good standby to prove a point is a second outage,
    #    so try that first.
    #
    #    A PROMOTED node is different in kind: it switched to its own WAL
    #    timeline the moment it was promoted, so it can never reattach to the
    #    old primary. That one has to be re-cloned (production: pg_rewind).
    reattached = False
    if promoted is None:
        _run(["docker", "compose", "up", "-d", STANDBY_SERVICE], cwd=LAB_ROOT)
        reattached = wait_topology(timeout=60)
    if reattached:
        print("    ✅ standby reattached to the restarted primary — no re-clone needed")
    else:
        print("    …  re-cloning the standby from the primary (takes a moment)")
        rebuild_standby()
        print("    ✅ standby is streaming again")

    # 5. Verify. A failback you did not verify is a failback you did not do.
    assert node_role(PRIMARY_PORT) == "primary", "5432 is not a writable primary"
    assert node_role(STANDBY_PORT) == "standby", "55433 is not in recovery"
    assert streaming_ok(), "no standby is streaming from the primary"
    if markers:
        deadline, seen = time.time() + 30, 0
        while time.time() < deadline:
            rows = pg_query(
                STANDBY_PORT,
                "SELECT COUNT(*) FROM audit_log WHERE changed_by = ANY(%s)",
                (markers,))
            seen = rows[0][0] if rows else 0
            if seen >= len(markers):
                break
            time.sleep(1)
        assert seen >= len(markers), (
            f"only {seen}/{len(markers)} failover-window write(s) reached the "
            f"rebuilt standby -- the failback lost data it claimed to preserve")
        print(f"    ✅ all {len(markers)} failover-window write(s) survived the "
              f"failback and replicated to the new standby")

    print("✅ FAILBACK complete: primary on 5432, streaming standby on 55433.")
    return {"action": "failback", "carried_back": carried,
            "carried_markers": markers}


failback()


## 🛠️ Setup

The preflight cell above already restored the documented topology (writable
primary on 5432, streaming standby on 55433) whether you got here from a cold
start or from notebook 2's failover. The drill below **needs** that shape: it
measures data loss against a standby, so it cannot start with the standby
already promoted.

If the preflight reports it could not repair things, reset the lab:

```bash
cd 08-enterprise/bcdr
docker compose down -v && docker compose up -d --wait
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).


In [ ]:
import psycopg2
import subprocess
import time
import datetime
import json
from tabulate import tabulate

DB_PRIMARY = {
    "host": "localhost", "port": 5432,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}
DB_STANDBY = {
    "host": "localhost", "port": 55433,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def get_standby_connection():
    return psycopg2.connect(**DB_STANDBY)

def docker_exec(container, cmd, user=None):
    """`docker exec` runs as root unless told otherwise, and `pg_ctl` refuses
    to run as root ("cannot be run as root") because the server must never own
    files as root. Client tools (psql, pg_dump) do not care. Pass
    user="postgres" for the server-side ones."""
    prefix = ["docker", "exec"] + (["-u", user] if user else [])
    result = subprocess.run(
        prefix + [container] + cmd,
        capture_output=True, text=True, timeout=30
    )
    return result.stdout.strip(), result.stderr.strip()

def check_container(name):
    """Check if a Docker container is running."""
    result = subprocess.run(
        ["docker", "inspect", "-f", "{{.State.Running}}", name],
        capture_output=True, text=True
    )
    return result.stdout.strip() == 'true'

# Verify clean state. These are hard requirements, not FYIs: a drill run
# against the wrong topology produces numbers that look like measurements and
# are not.
assert node_role(5432) == "primary", (
    f"port 5432 is '{node_role(5432)}', not a writable primary -- re-run the "
    f"preflight cell at the top of this notebook")
assert node_role(55433) == "standby", (
    f"port 55433 is '{node_role(55433)}', not a standby in recovery -- the drill "
    f"below has nothing to fail over to")
print("✅ Primary is running and writable (5432)")
print("✅ Standby is running and in recovery (55433)")

## 📚 DR Drill Plan

A proper DR drill has these phases:

```
Phase 1: PRE-DRILL HEALTH CHECK
  - Verify all services are running
  - State the RPO and RTO objectives BEFORE you know the result
  - Record baseline data counts

Phase 2: SIMULATE DISASTER
  - Write data while replication is healthy (should survive)
  - Break the replication stream, write more (should NOT survive)
  - Kill the primary database; record the exact time of failure

Phase 3: DETECT AND RESPOND
  - Detect that the primary is down
  - Check which of those last-moment writes reached the standby
  - Turn that into an RPO measured in TIME

Phase 4: EXECUTE FAILOVER
  - Promote standby to primary
  - Verify the new primary accepts writes
  - Measure the mechanical failover time (the RTO)

Phase 5: POST-DRILL ANALYSIS
  - Grade actual RPO and RTO against the objectives from phase 1
  - Generate the drill report

Phase 6: FAILBACK
  - Rebuild the fenced node, re-form the pair, and prove the writes made
    during the outage were not lost
```

### The two numbers, kept apart

**RPO is a time**, not a row count. "We lost 3 records" is a symptom; the
objective is expressed as "the recovery point may not sit more than N seconds
behind the last committed write". Phase 3 measures that gap directly.

**RTO is also a time**, but a completely different one: how long the system
could not accept writes. A drill can meet its RTO and blow its RPO, or the
reverse. Reporting one number for both is the most common way DR reports lie.


In [ ]:
# =============================================================================
# Phase 1: PRE-DRILL HEALTH CHECK
# =============================================================================

drill_log = {}  # We will record everything here
drill_log["drill_start"] = datetime.datetime.now().isoformat()

# State the objectives BEFORE the drill. A drill without a declared target is
# a story, and a target invented afterwards is always met.
#
#   RPO target: how far back the recovery point may sit behind the last
#               committed write. The business asked for zero.
#   RTO target: how long the system may be unable to accept writes.
RPO_TARGET_SECONDS = 0.0
RTO_TARGET_SECONDS = 30.0
drill_log["rpo_target_seconds"] = RPO_TARGET_SECONDS
drill_log["rto_target_seconds"] = RTO_TARGET_SECONDS

print("=" * 72)
print("PHASE 1: PRE-DRILL HEALTH CHECK")
print("=" * 72)
print("\nObjectives declared up front:")
print(f"  RPO target: {RPO_TARGET_SECONDS:.0f} s of committed data may be lost")
print(f"  RTO target: {RTO_TARGET_SECONDS:.0f} s until writes are accepted again")

# Check replication status
primary_conn = get_primary_connection()
primary_conn.autocommit = True
primary_cur = primary_conn.cursor()

primary_cur.execute(
    "SELECT client_addr, state, sync_state FROM pg_stat_replication"
)
repl = primary_cur.fetchall()
assert repl, (
    "no standby is streaming from the primary -- ABORT the drill. Re-run the "
    "preflight cell at the top of this notebook. Running the drill without a "
    "standby would 'measure' a failover that cannot happen.")
drill_log["sync_state"] = repl[0][2]
print(f"\n✅ Replication active: {repl[0][1]} ({repl[0][2]} mode)")
print("   Note 'async': the primary acknowledges commits before the standby")
print("   has them. That is structurally incompatible with an RPO of 0, and")
print("   phase 3 will show exactly what it costs.")

# Record baseline data
tables = ['customers', 'orders', 'order_items', 'payments']
baseline = {}
for tbl in tables:
    primary_cur.execute(f"SELECT COUNT(*) FROM {tbl}")
    baseline[tbl] = primary_cur.fetchone()[0]

drill_log["baseline_counts"] = baseline

print("\nBaseline data counts:")
for tbl, count in baseline.items():
    print(f"  {tbl}: {count} rows")

# Record primary WAL position
primary_cur.execute("SELECT pg_current_wal_lsn()")
primary_lsn = str(primary_cur.fetchone()[0])
drill_log["primary_lsn_before_disaster"] = primary_lsn
print(f"\nPrimary WAL position: {primary_lsn}")
print("\n✅ Phase 1 complete. System is healthy.")

primary_conn.close()


In [ ]:
# =============================================================================
# Phase 2: SIMULATE DISASTER
# =============================================================================
# Two batches of last-moment writes:
#
#   replicated_* — written while the standby is up and streaming. These
#                  survive the crash.
#   orphan_*     — written while the standby is gone. The primary
#                  ACKNOWLEDGED them; the standby never saw them. These are
#                  the data loss, and the gap between the batches IS the RPO.
#
# How we take the standby out matters. Two tempting mechanisms do NOT give a
# reliable result:
#
#   * terminating the walsender — the standby simply reconnects after
#     `wal_retrieve_retry_interval` (5s), so whether anything is lost is a
#     race against docker, and the drill reports RPO 0 about half the time;
#   * `docker pause` — the WAL still lands in the frozen standby's socket
#     receive buffer, and it reads it the moment it resumes.
#
# Stopping the container is provably total: a stopped PostgreSQL receives
# nothing, buffers nothing, and when it comes back its only source of the
# missing WAL is a primary that no longer exists. And it is not contrived —
# "the standby was down for maintenance / had crashed / was still catching up
# when the primary died" is one of the most common ways real clusters discover
# what their RPO actually is.
#
# Note what SYNCHRONOUS replication would have done here: with
# synchronous_standby_names set, the orphan writes below would have BLOCKED
# rather than returned "committed". Nothing would be lost, because nothing
# would have been promised. That is the whole trade.

print("=" * 72)
print("PHASE 2: SIMULATE DISASTER")
print("=" * 72)

primary_conn = get_primary_connection()
primary_conn.autocommit = True
primary_cur = primary_conn.cursor()

written_at = {}       # audit_log id -> wall clock when the commit returned
all_ids, replicated_ids, orphan_ids = [], [], []


def write_marker(tag):
    """Commit one audit row and remember exactly when the commit came back."""
    primary_cur.execute(
        "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
        "VALUES (%s, %s, %s, %s) RETURNING id",
        ('dr_drill', len(all_ids), 'INSERT', tag)
    )
    rid = primary_cur.fetchone()[0]
    written_at[rid] = time.time()
    all_ids.append(rid)
    return rid


print("\nWriting last-moment data while replication is healthy...")
for i in range(5):
    replicated_ids.append(write_marker(f'replicated_{i}'))
    time.sleep(0.1)
print(f"  wrote {replicated_ids} — these should survive the crash")

# Wait until the standby genuinely has them, so 'survived' in phase 3 is a
# fact rather than a race we happened to win.
standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()
deadline = time.time() + 15
replicated_seen = 0
while time.time() < deadline:
    standby_conn.rollback()
    standby_cur.execute("SELECT COUNT(*) FROM audit_log WHERE id = ANY(%s)",
                        (replicated_ids,))
    replicated_seen = standby_cur.fetchone()[0]
    if replicated_seen == len(replicated_ids):
        break
    time.sleep(0.05)
assert replicated_seen == len(replicated_ids), (
    f"the standby only has {replicated_seen}/{len(replicated_ids)} of the "
    f"healthy writes -- replication is broken before the drill even starts")
print(f"  ✅ standby has all {replicated_seen} of them")
standby_conn.close()

# --- take the standby out of the picture, provably --------------------------
print("\nThe standby drops out (maintenance, crash, long partition)...")
subprocess.run(["docker", "stop", "-t", "30", STANDBY_CONTAINER],
               capture_output=True, timeout=90)

assert not port_open(STANDBY_PORT), (
    f"the standby is still listening on {STANDBY_PORT} -- it did not stop, so "
    f"the writes below might still replicate and the drill would measure "
    f"nothing")
primary_cur.execute("SELECT COUNT(*) FROM pg_stat_replication")
attached = primary_cur.fetchone()[0]
assert attached == 0, (
    f"the primary still reports {attached} attached standby(s) after stopping "
    f"the container")
print("  ✅ standby is DOWN and detached — the primary is now alone")

# --- writes the standby can never see ---------------------------------------
for i in range(3):
    orphan_ids.append(write_marker(f'orphan_{i}'))
print(f"  wrote {orphan_ids} with NO standby attached — these are the loss")
print("     (the primary answered 'committed' to all three)")

drill_log["replicated_record_ids"] = replicated_ids
drill_log["orphan_record_ids"] = orphan_ids
drill_log["all_record_ids"] = all_ids

# === DISASTER STRIKES ===
print("\n💥 SIMULATING PRIMARY FAILURE (SIGKILL — no clean shutdown)...\n")
disaster_time = time.time()
drill_log["disaster_time"] = datetime.datetime.now().isoformat()
primary_conn.close()

subprocess.run(["docker", "kill", PRIMARY_CONTAINER],
               capture_output=True, timeout=30)

primary_down = False
try:
    psycopg2.connect(connect_timeout=3, **DB_PRIMARY).close()
except psycopg2.Error:
    primary_down = True
assert primary_down, (
    "the primary is still accepting connections -- the disaster did not "
    "happen, so nothing below is a recovery")
print("✅ Primary is DOWN.")

# --- bring the standby back: it is all we have left -------------------------
# In a real incident this boot time counts toward your RTO, and it is why a
# standby you keep switched off is a cold standby with a warm price tag.
print("\nBringing the standby back up — it is the only copy left...")
subprocess.run(["docker", "start", STANDBY_CONTAINER],
               capture_output=True, timeout=60)
deadline = time.time() + 120
while time.time() < deadline:
    if node_role(STANDBY_PORT) == "standby":
        break
    time.sleep(2)
assert node_role(STANDBY_PORT) == "standby", (
    "the standby did not come back in recovery mode -- there is nothing left "
    "to fail over to")
print("✅ Standby is back, still in recovery, and cannot reach its dead primary.")
print(f"   Time of failure: {drill_log['disaster_time']}")


In [ ]:
# =============================================================================
# Phase 3: DETECT AND RESPOND
# =============================================================================

print("=" * 72)
print("PHASE 3: DETECT AND RESPOND")
print("=" * 72)

detection_start = time.time()
try:
    psycopg2.connect(connect_timeout=3, **DB_PRIMARY).close()
    detected = False
except psycopg2.Error:
    detected = True
detection_time = time.time() - detection_start
drill_log["detection_time_seconds"] = detection_time

assert detected, "the primary answered -- there is nothing to recover from"
print(f"\n  ❌ Primary is unreachable (this probe returned in {detection_time:.3f}s)")
print("     Treat that as a FLOOR, not a detection time. A real monitor")
print("     notices a failure after (check interval x failure threshold),")
print("     typically 5-30s, and every second of it is part of your RTO.")

# Check standby health
standby_conn = get_standby_connection()
standby_cur = standby_conn.cursor()
standby_cur.execute("SELECT pg_is_in_recovery()")
assert standby_cur.fetchone()[0], (
    "the standby is not in recovery -- it was already promoted, so there is "
    "no failover left to perform or measure")
print("\n  ✅ Standby is running and still in recovery mode")

standby_cur.execute("SELECT pg_last_wal_receive_lsn(), pg_last_wal_replay_lsn()")
recv_lsn, replay_lsn = standby_cur.fetchone()
print(f"  Last WAL received:  {recv_lsn}")
print(f"  Last WAL replayed:  {replay_lsn}")
print("  (Received can read LOWER than replayed here: the standby was just")
print("   restarted, so its WAL receiver has not started a new stream — there")
print("   is no primary left to stream from. Replayed is the number that")
print("   matters; it is the recovery point.)")

# Which of the last-moment writes actually made it across?
print("\n  Checking last-moment records on standby...")
standby_cur.execute(
    "SELECT id FROM audit_log WHERE id = ANY(%s) ORDER BY id", (all_ids,))
survived_ids = [r[0] for r in standby_cur.fetchall()]
lost_ids = [i for i in all_ids if i not in survived_ids]
standby_conn.close()

# --- turn that into an RPO, which is a TIME ---------------------------------
# The recovery point is the last write that made it across. Everything
# committed after that instant is gone, so the RPO achieved is the gap
# between the last commit the primary acknowledged and that recovery point.
# If nothing was lost the RPO is 0 by definition -- NOT "time since the last
# write", which is the classic way this number gets inflated.
last_written_at = max(written_at.values())
if not lost_ids:
    rpo_seconds = 0.0
elif survived_ids:
    rpo_seconds = last_written_at - max(written_at[i] for i in survived_ids)
else:
    # Nothing survived: we can only say the loss is at least this wide.
    rpo_seconds = last_written_at - min(written_at.values())

drill_log["records_total"] = len(all_ids)
drill_log["records_survived"] = len(survived_ids)
drill_log["records_lost"] = len(lost_ids)
drill_log["rpo_seconds"] = rpo_seconds

print(f"  Writes acknowledged by the primary: {len(all_ids)}")
print(f"  Present on the standby:             {len(survived_ids)} {survived_ids}")
print(f"  LOST:                               {len(lost_ids)} {lost_ids}")
print(f"\n  → RPO achieved: {rpo_seconds*1000:.0f} ms of committed writes are gone")

# The standby was provably stopped for the whole window in which the orphan
# writes were made, so this is not a race and there is no "sometimes" branch:
# exactly those writes must be gone, and nothing else may be.
assert set(lost_ids) == set(orphan_ids), (
    f"expected exactly the writes made with no standby attached "
    f"({orphan_ids}) to be lost, got {lost_ids} -- either the drill is no "
    f"longer demonstrating that async replication implies RPO > 0, which is "
    f"its entire point, or it lost data it did not intend to")
assert set(survived_ids) == set(replicated_ids), (
    f"expected every write made while the standby was streaming "
    f"({replicated_ids}) to survive, got {survived_ids}")
assert rpo_seconds > 0, "RPO must be greater than zero once writes were lost"
print("  ✅ Exactly the writes made with no standby attached were lost —")
print("     and every write made while it was streaming survived.")


In [ ]:
# =============================================================================
# Phase 4: EXECUTE FAILOVER
# =============================================================================

print("=" * 72)
print("PHASE 4: EXECUTE FAILOVER")
print("=" * 72)

failover_start = time.time()

print("\nPromoting standby to primary...")
out, err = docker_exec(
    "bcdr-pg-standby",
    ["pg_ctl", "promote", "-D", "/var/lib/postgresql/data"],
    user="postgres",   # pg_ctl refuses to run as root
)
print(f"  pg_ctl output: {out or err}")

# Poll for the promotion instead of sleeping a fixed number of seconds. A
# hard-coded sleep would be baked straight into the RTO reported below, and
# the notebook would be 'measuring' a constant it chose itself.
new_primary_conn = None
deadline = time.time() + 60
while time.time() < deadline:
    try:
        conn = psycopg2.connect(connect_timeout=2, **DB_STANDBY)
        conn.autocommit = True   # set BEFORE any query opens a transaction
        cur = conn.cursor()
        cur.execute("SELECT pg_is_in_recovery()")
        if not cur.fetchone()[0]:
            new_primary_conn = conn
            break
        conn.close()
    except psycopg2.Error:
        pass
    time.sleep(0.1)

assert new_primary_conn is not None, (
    "the standby never left recovery mode -- promotion failed, and the drill "
    "has no recovered system to measure")
print("  ✅ Standby promoted — no longer in recovery mode!")

new_cur = new_primary_conn.cursor()
new_cur.execute(
    "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
    "VALUES (%s, %s, %s, %s) RETURNING id",
    ('dr_drill', 999, 'FAILOVER_TEST', 'new_primary_write_test')
)
failover_write_id = new_cur.fetchone()[0]

failover_only_time = time.time() - failover_start
wall_clock_rto = time.time() - disaster_time
new_primary_conn.close()

drill_log["failover_time_seconds"] = failover_only_time
drill_log["wall_clock_rto_seconds"] = wall_clock_rto
drill_log["failover_write_id"] = failover_write_id

print(f"  ✅ Write #{failover_write_id} succeeded on the new primary.")
print(f"\n  ⏱️  Mechanical failover (promote → writable): {failover_only_time:.2f} s")
print(f"  ⏱️  Wall clock since the crash:               {wall_clock_rto:.1f} s")
print("      The wall-clock figure includes however long YOU took to read")
print("      phase 3 and press run. That human time is real and it belongs in")
print("      a manual-failover RTO — it is the whole argument for Patroni.")
print("      The two numbers are graded separately in phase 5 for that reason.")

assert failover_only_time < 120, (
    f"promotion took {failover_only_time:.0f}s -- something hung; that is not "
    f"a credible hot-standby failover time")


In [ ]:
# =============================================================================
# Phase 5: POST-DRILL ANALYSIS
# =============================================================================

# Verify data integrity on the new primary
new_conn = psycopg2.connect(**DB_STANDBY)
new_cur = new_conn.cursor()

post_counts = {}
for tbl in ['customers', 'orders', 'order_items', 'payments']:
    new_cur.execute(f"SELECT COUNT(*) FROM {tbl}")
    post_counts[tbl] = new_cur.fetchone()[0]

new_conn.close()

rpo = drill_log["rpo_seconds"]
rto = drill_log["failover_time_seconds"]
wall_rto = drill_log["wall_clock_rto_seconds"]
rpo_target = drill_log["rpo_target_seconds"]
rto_target = drill_log["rto_target_seconds"]
records_lost = drill_log["records_lost"]

print("\n" + "=" * 72)
print("        DISASTER RECOVERY DRILL REPORT")
print("=" * 72)
print(f"\n  Drill Start:    {drill_log['drill_start']}")
print(f"  Disaster Time:  {drill_log['disaster_time']}")
print(f"  Replication:    {drill_log['sync_state']}")

print("\n--- RTO: how long we could not accept writes ---")
print(f"  Failure-detection probe:    {drill_log['detection_time_seconds']:.3f} s  (a floor, see phase 3)")
print(f"  Mechanical failover:        {rto:.2f} s")
print(f"  Wall clock incl. operator:  {wall_rto:.1f} s")
print(f"  Target:                     {rto_target:.0f} s"
      f"  →  {'✅ MET' if rto <= rto_target else '❌ MISSED'} (mechanical)")

print("\n--- RPO: how much data we lost, expressed as time ---")
print(f"  Writes acknowledged by the primary: {drill_log['records_total']}")
print(f"  Present on the recovered node:      {drill_log['records_survived']}")
print(f"  Lost:                               {records_lost}")
print(f"  Recovery point sits behind the last commit by: {rpo*1000:.0f} ms")
print(f"  Target:                     {rpo_target:.0f} s"
      f"  →  {'✅ MET' if rpo <= rpo_target else '❌ MISSED'}")

print("\n--- Data Integrity (tables the drill never touched) ---")
table_data = []
for tbl in ['customers', 'orders', 'order_items', 'payments']:
    orig = baseline[tbl]
    post = post_counts[tbl]
    table_data.append([tbl, orig, post, "✅" if orig == post else "⚠️"])

print(tabulate(table_data,
    headers=["Table", "Before Disaster", "After Recovery", "Match"],
    tablefmt="grid"))

print("\n--- Verdict ---")
if rto <= rto_target:
    print(f"  ✅ RTO {rto:.2f}s beats the {rto_target:.0f}s objective.")
else:
    print(f"  ❌ RTO {rto:.2f}s misses the {rto_target:.0f}s objective.")
print(f"     (Wall clock was {wall_rto:.1f}s. Automating the promotion is what")
print("      closes the gap between the two.)")

if rpo <= rpo_target:
    print(f"  ✅ RPO {rpo*1000:.0f} ms meets the {rpo_target:.0f}s objective.")
else:
    print(f"  ❌ RPO {rpo*1000:.0f} ms MISSES the {rpo_target:.0f}s objective — "
          f"{records_lost} acknowledged writes are gone.")
    print("     This is not a bug in the drill; it is what asynchronous")
    print("     replication means. The primary said 'committed' before the")
    print("     standby had the WAL. The only fix that actually gets you to")
    print("     RPO 0 is synchronous_commit = on with synchronous_standby_names")
    print("     set — and you pay for it on every single write, which is the")
    print("     latency you measured back in notebook 1.")

# The drill only ever wrote to audit_log. If any other table moved, the
# failover lost or invented rows and the report above is fiction.
assert post_counts == baseline, (
    f"tables the drill never touched changed across the failover: "
    f"{baseline} -> {post_counts}")

print("\n" + "=" * 72)
print("  Phase 6 below puts the cluster back together. Do not stop here:")
print("  right now there is no standby at all, and the next failure has")
print("  nothing to fail over to.")
print("=" * 72)


## 📚 Phase 6: Failback — the half of the runbook everyone skips

The drill is not over when the standby is writable. It is over when the
cluster is back in a shape that could survive the **next** failure.

Right now it is not:

- port 55433 is a lone primary with no replica,
- port 5432 is a corpse holding transactions nobody can read, and
- the write we made in phase 4 exists on exactly one node.

Failback has to deal with all three, and the middle one is where teams lose
data. The fenced node and the promoted node have **diverged**: each holds
commits the other has never seen. Restarting the fenced node without
reconciling that divergence gives you two primaries with different histories —
split-brain after the fact — and re-cloning the promoted node from the fenced
one throws the outage's writes away silently.

The `failback()` helper defined in the preflight cell captures the promoted
node's writes first, brings the fenced node back, replays them onto it, and
only then re-clones the standby. Below we check it actually did that.


In [ ]:
# =============================================================================
# Phase 6: FAILBACK — put the cluster back the way it was
# =============================================================================

print("=" * 72)
print("PHASE 6: FAILBACK")
print("=" * 72)

drill_log["failback"] = failback()

# The write we made on the promoted node in phase 4 must still exist. A
# failback that loses it is silent data loss, and it is precisely what this
# phase exists to catch -- "the failover worked" is not the same claim as
# "nothing written during the outage was lost".
rows = pg_query(PRIMARY_PORT,
                "SELECT COUNT(*) FROM audit_log WHERE changed_by = %s",
                ('new_primary_write_test',))
carried = rows[0][0] if rows else 0
assert carried >= 1, (
    "the write made on the promoted node during the outage did not survive "
    "the failback -- that is silent data loss dressed up as a clean recovery")
print(f"\n✅ The failover-window write survived the failback ({carried} row).")

# And the writes the crash 'lost'? They were fsynced to the old primary's disk
# before it died -- they were simply unreadable while it was down, which is
# exactly what made them an RPO breach. Restarting the node recovers them from
# that disk. That is forensic cleanup, not a smaller RPO.
rows = pg_query(PRIMARY_PORT,
                "SELECT changed_by FROM audit_log "
                "WHERE changed_by LIKE 'orphan\\_%' ORDER BY changed_by")
recovered = [r[0] for r in (rows or [])]
print(f"✅ Recovered from the fenced node's disk: {recovered or 'none'}")
print("   This does NOT lower the drill's RPO. Those rows were invisible for")
print("   the entire outage — customers and downstream systems already saw")
print("   them as lost. Production `pg_rewind` would DISCARD them outright as")
print("   diverged history; reconciling them is a manual, forensic job.")

assert topology_ok(), (
    "the lab is not back in a runnable state: expected a writable primary on "
    "5432 with a streaming standby on 55433")
print("\n🎉 Cluster restored: writable primary on 5432, streaming standby on 55433.")
print("   Notebooks 1-4 can be re-run from here without a docker reset —")
print("   which is the point: a drill you cannot repeat is a drill you will")
print("   only ever run once.")


## 📚 Redis Failover

Our lab also includes Redis replication. Let us check its status.
In production, you would use **Redis Sentinel** or **Redis Cluster** for automatic failover.

In [ ]:
# =============================================================================
# Demo: Check Redis Replication Status
# =============================================================================

import redis

r_primary = redis.Redis(host='localhost', port=6379, decode_responses=True)
r_replica = redis.Redis(host='localhost', port=6380, decode_responses=True)

print("=" * 65)
print("REDIS REPLICATION STATUS")
print("=" * 65)

# Write to primary, then POLL the replica. Redis replication is asynchronous
# too, so a fixed sleep is either wasteful or flaky -- the same trap as the
# PostgreSQL side.
r_primary.set("bcdr:test:key", "hello from primary")
deadline = time.time() + 10
replica_val = None
while time.time() < deadline:
    replica_val = r_replica.get("bcdr:test:key")
    if replica_val == "hello from primary":
        break
    time.sleep(0.05)

# Primary info
info = r_primary.info('replication')
print(f"\nPrimary role:          {info['role']}")
print(f"Connected replicas:    {info['connected_slaves']}")

# Replica info
rinfo = r_replica.info('replication')
print(f"\nReplica role:          {rinfo['role']}")
print(f"Master host:           {rinfo.get('master_host', 'N/A')}")
print(f"Master link status:    {rinfo.get('master_link_status', 'N/A')}")

print(f"\nReplication test:")
print(f"  Wrote to primary:    'hello from primary'")
print(f"  Read from replica:   '{replica_val}'")

# Cleanup
r_primary.delete("bcdr:test:key")

assert info['role'] == 'master' and rinfo['role'] == 'slave', (
    f"unexpected Redis roles: primary={info['role']}, replica={rinfo['role']}")
assert rinfo.get('master_link_status') == 'up', (
    f"the replica's link to the primary is "
    f"'{rinfo.get('master_link_status')}', not 'up'")
assert replica_val == 'hello from primary', (
    f"the replica returned {replica_val!r} -- Redis replication is not working, "
    f"so the cache tier has no failover story at all")
print("  ✅ Redis replication is working!")
print("\n💡 Redis replication is async by default as well, so the same RPO")
print("   caveat applies: a promoted replica can be missing the last writes.")
print("   Sentinel automates the promotion; it does not make it lossless.")


## 📝 Summary

### What You Accomplished in This Drill

1. **Pre-drill check** — Verified replication health, recorded baseline data,
   and declared the RPO and RTO objectives *before* seeing any result.
2. **Simulated disaster** — Wrote data with the stream healthy, broke the
   stream, wrote more, then SIGKILLed the primary.
3. **Detected failure** — Confirmed the primary was unreachable and measured
   which acknowledged writes never reached the standby.
4. **Executed failover** — Promoted the standby, verified writes, and timed
   the promotion by polling rather than by sleeping.
5. **Measured results** — Graded actual RPO (a time) and RTO (a different
   time) against the declared objectives.
6. **Failed back** — Rebuilt the fenced node, re-formed the pair, and proved
   the write made during the outage was not lost.

### Key Takeaways

1. **RPO and RTO are two different clocks.** One measures data, one measures
   downtime, and a system can pass either while failing the other. Any report
   that gives you a single "recovery number" is hiding one of them.
2. **Async replication cannot promise RPO = 0.** Phase 3 shows the cost in
   milliseconds of acknowledged, lost writes. Synchronous replication removes
   it and charges you on every write instead.
3. **Measure, do not sleep.** Every fixed `sleep()` in a failover script ends
   up inside the RTO you report.
4. **Practice makes perfect** — The first time you failover should NOT be
   during a real disaster.
5. **Failback is part of the drill.** A cluster that has failed over has no
   standby left, and the promoted node holds writes nobody else has.
6. **Test regularly** — Run DR drills quarterly at minimum. Things change:
   new data, new services, new people.
7. **Document the procedure** — Can someone else do this at 3 AM on a Saturday?

### What this lab does NOT do

- **No automatic failover.** Every promotion here was a human running a cell.
  Patroni/pg_auto_failover add consensus, leader leases and fencing so a
  machine decides in seconds.
- **No quorum.** With two nodes there is no majority to appeal to; we
  prevented split-brain by fencing by hand. Real clusters run 3 or 5 nodes.
- **No off-site copy.** Every byte here lives on one laptop. The 3-2-1 rule
  in notebook 3 is described, not demonstrated.
- **Row-level failback.** `failback()` carries the outage's writes across at
  row level because this lab confines them to one table. Production uses
  `pg_rewind` plus WAL replay, which handles every table and preserves row
  identity.

### Enterprise BCDR Maturity Model

| Level | Description | This Lab |
|-------|------------|----------|
| 1 - Ad hoc | No plan, no backups | - |
| 2 - Backup | Regular backups, untested | Notebook 3 |
| 3 - Replication | Hot standby, manual failover | Notebook 2 |
| 4 - Automated | Automatic failover, monitored | Next step |
| 5 - Chaos | Regular DR drills, chaos engineering | This notebook! |

### What is Next?

To take BCDR further, explore:
- **Patroni** — Automatic PostgreSQL failover with consensus
- **pg_rewind** — Re-attach a diverged old primary without a full re-clone
- **Redis Sentinel** — Automatic Redis failover
- **Chaos Engineering** — Netflix Chaos Monkey, Gremlin, Litmus
- **Multi-region** — Azure Paired Regions, AWS Multi-AZ
- **Runbooks** — Documented step-by-step procedures for every failure scenario
